# The Event Horizon Telescope

### How a planet-sized telescope works, and what it actually records

Notebook 01 ended with a target: a polarized image of a ring about **42
microarcseconds** across. That is roughly an orange on the Moon, and no telescope
that has been built, or could be built, resolves it.

The EHT gets there by not being a telescope. It is eight (now eleven) separate
dishes scattered across the planet, recording onto hard drives with atomic clocks
for timestamps, combined months later in a supercomputer. What comes out is not a
picture. It is a sparse handful of complex numbers, one per pair of dishes per
moment, and turning those into an image is a problem with no unique answer.

**What it covers**

1. why a pair of dishes measures one *Fourier component* of the sky rather than a
   piece of the picture;
2. how the Earth's rotation fills in the gaps, and how to read a $uv$ coverage
   plot for what the array can and cannot know;
3. what a single station physically records: two feed voltages, on a mount that
   turns against the sky;
4. why making an image from those numbers requires assumptions, and what survives
   when the calibration does not.

**Prerequisites:** notebook 01, and the idea of a Fourier transform.

It is the longest of the four notebooks, in four parts, and each part stands on
its own.

In [2]:
# Every function this notebook calls lives in ../scripts.
import sys
from pathlib import Path

SCRIPTS = Path.cwd() / "scripts"
if not SCRIPTS.is_dir():
    SCRIPTS = Path.cwd().parent / "scripts"
sys.path.append(str(SCRIPTS))
print("importing from", SCRIPTS)

import arrays
import interactive_telescope as explore
import interferometry as itf
import numpy as np

importing from /mnt/c/Users/alexw/OneDrive - University of Toronto/SUMMER_26/EHT/CSC494-report/scripts


---
# Part A · Two dishes and a clock

## The telescope you would need is the size of the Earth

A telescope of diameter $D$ observing at wavelength $\lambda$ cannot resolve
anything finer than about $\lambda/D$ radians. That's called diffraction.

The EHT observes at $\lambda = 1.3$ mm, a wavelength chosen because the plasma
around the black hole becomes transparent there, and hated by everyone involved
because water vapour absorbs it (hence the volcanoes and deserts). Ask the
formula what aperture 42 μas requires:

In [6]:
print(f"a 12 m dish at 1.3 mm:        {itf.resolution_uas(1.3e-3, 12):>12,.0f} µas")
print(f"the 300 m Arecibo dish:       {itf.resolution_uas(1.3e-3, 300):>12,.0f} µas")
print(f"needed for M87's 42 µas ring: {itf.required_aperture_m(1.3e-3, 42) / 1e3:>12,.0f} km")
print("the Earth is                        12,742 km across"
      "   <- just enough, with nothing to spare")

a 12 m dish at 1.3 mm:          22,345,354 µas
the 300 m Arecibo dish:            893,814 µas
needed for M87's 42 µas ring:        6,384 km
the Earth is                        12,742 km across   <- just enough, with nothing to spare


Half of that, and the factor of two is not spare capacity. It is the whole
imaging budget. At exactly 6,384 km the ring would be *one* resolution element
across, which is another way of saying unresolved; making out that it is a ring,
with a brighter side, takes several. And the full diameter is never on offer: two
dishes only work as a pair while *both* can see the source, which rules out
anything near an antipodal pair, and what sets the resolution is not the distance
between them but that distance **projected onto the plane of the sky**, which is
always shorter.

Part B measures what the 2017 array actually managed: 10,907 km projected, which
works out to a resolution of 24.7 μas against a ring 42 μas across. Two
resolution elements.
That is what "nothing to spare" means.

The explorer below is the same formula, to play with:

In [25]:
explore.resolution_explorer()

So the aperture has to be the size of the planet. Fortunately, we don't need
the whole aperture, we only need *pairs of points* on it.

## Two dishes make a comb of fringes, not a picture

Take two dishes separated by a distance $b$, pointed at the same source, and
multiply their voltages together. That pair of dishes, together with the vector
between them, is a **baseline**, and the machine that multiplies the two
recordings together is the **correlator**, which for the EHT is a supercomputer
working months later on hard drives that were flown in. Doing interferometry this way,
between dishes far too distant to be wired together, is what **very long baseline
interferometry** (VLBI) means.

Light from a source slightly off-axis arrives at one dish before the other, and
that geometric delay makes the multiplied signal oscillate as a function of
position on the sky.

A pair of dishes is therefore not a camera. It's a **comb of fringes** laid
across the sky, with angular spacing $\lambda/b$, and the correlator reports how
much of the source lines up with the comb.

In [7]:
explore.fringe_explorer()

Push the baseline down to a few tens of kilometres: the fringes get so wide that
the entire ring sits inside one bright stripe, and the response saturates. That
baseline learns the source's *total flux* and nothing else. Push it out to
thousands of kilometres and the comb becomes finer than the ring, so the bright
and dark stripes cut across the source and the response starts to depend on the
details.

## What a pair of dishes actually measures

Make that precise. Write the baseline in units of the observing wavelength and
project it onto the plane of the sky; call the two components $(u, v)$. Then the
correlator output, the **visibility**, is

$$V(u, v) = \int I(l, m)\; e^{-2\pi i (ul + vm)}\, \mathrm{d}l\, \mathrm{d}m$$

which is the two-dimensional Fourier transform of the sky brightness,
**evaluated at exactly one point.** This is the van Cittert–Zernike theorem, and
it is the reason any of this works.

So: baseline *length* sets the fringe spacing, baseline *orientation* sets the
fringe direction, and the correlator hands back one complex number saying how
strongly the sky resembles that pattern. Drag a baseline around and watch:

In [27]:
explore.fourier_explorer()

Two things worth noticing.

**The amplitude collapses at particular baseline lengths.** When the fringe
spacing matches the ring's diameter, the bright and dark stripes cover equal
amounts of the ring and the contributions cancel. That null is a direct
measurement of the ring's size, and it is how the 42 μas number was pinned down.

**The phase is doing something too.** Shift the source on the sky and the
amplitude is unchanged while the phase winds; a symmetric source has zero phase,
and asymmetry (a brighter side of the ring) shows up as phase structure. Hold
onto that: it comes back in Part D, because the phase is also the part that
calibration destroys.

## Reading a source's size off the curve

Plot amplitude against baseline length and the source's size is legible directly:

In [28]:
explore.visibility_profile_explorer()

A smooth Gaussian blob has no null at all. Its visibility just falls away
monotonically, because a Gaussian's Fourier transform is a Gaussian. Rings and
sharp edges put oscillations into the curve, and the first deep minimum sits
near $u \approx 1.22/d$ for a ring of diameter $d$. For 42 μas that lands around
3.4 Gλ, which is a baseline of about 4,400 km. (Baselines are quoted in
wavelengths rather than metres, because it is the number of wavelengths that sets
the fringe spacing. One Gλ is a billion wavelengths, about 1,300 km at 1.3 mm.)

**The EHT's station list was chosen so that baselines of that length exist.** The
instrument was designed around the number it wanted to measure.

---
# Part B · An array on a spinning planet

## Where the dishes are

Eight stations observed M87 in April 2017. They're on volcanoes in Hawaii,
in the Chilean desert, in the Sierra Nevada, in Arizona, in Mexico, and at the
South Pole. Here they are with their real coordinates:

In [3]:
array = arrays.array_2017()
for code, site in sorted(array.items(), key=lambda kv: kv[1].sefd_jy):
    print(f"{code:5s} {site.name[:38]:40s} {site.feeds.upper()} feeds   "
          f"SEFD {site.sefd_jy:>5.0f} Jy   {np.rad2deg(site.latitude):+6.1f}° lat")

ALMA  Atacama Large Millimeter/submillimeter   XY feeds   SEFD    90 Jy    -22.9° lat
LMT   Large Millimeter Telescope Alfonso Ser   RL feeds   SEFD   600 Jy    +18.9° lat
PV    IRAM 30m Telescope, Pico Veleta          RL feeds   SEFD  1400 Jy    +36.9° lat
APEX  Atacama Pathfinder Experiment            RL feeds   SEFD  3500 Jy    -22.9° lat
SMA   Submillimeter Array                      RL feeds   SEFD  4900 Jy    +19.7° lat
SMT   Submillimeter Telescope                  RL feeds   SEFD  5000 Jy    +32.5° lat
SPT   South Pole Telescope                     RL feeds   SEFD  5000 Jy    -90.0° lat
JCMT  James Clerk Maxwell Telescope            RL feeds   SEFD  6000 Jy    +19.7° lat


Two things in that table matter later. The **SEFD** column is sensitivity: the
system equivalent flux density is how bright a source would have to be to be as
loud as the receiver's own noise, so smaller is better. It is quoted in
**janskys** (Jy), radio astronomy's unit of flux. For scale, the whole M87 ring
is under one jansky. ALMA's 90 Jy against everyone else's thousands is why ALMA
changed the EHT from a marginal experiment into an imaging instrument.

And the **feeds** column: ALMA records a *linear* (X/Y) basis while every other
station records a *circular* (R/L) one. That's the mixed-polarization problem
from notebook 01, sitting in the array table. Park it until Part C.

The globe below is centred on the point where the source is at the zenith, so
**a station you can see is a station that can see M87.** Run the clock:

In [5]:
explore.array_explorer()

Each grey arc is a live baseline, and the honest picture of what one *is* sits
underneath the drawing: not an arc across the surface but a straight chord
through several thousand kilometres of rock, connecting two dishes that are, from
the source's point of view, two points on one enormous aperture.

Watch the two places where two labels share one point. On Chajnantor in Chile,
ALMA and APEX are 2.6 km apart, and because ALMA is the array's only linear-feed
station, that pair is every mixed baseline's lower limit, the shortest one in
the array. On Mauna Kea, SMA and JCMT are 164 m apart, the shortest baseline of all.

Note also that the South Pole never appears. M87 sits at a declination of
$+12°$, declination being latitude on the sky, so from the pole it is permanently
below the horizon. SPT observed Sgr A\* that week instead; switch the source
dropdown and watch the array change.

## Who is on the air, and when

A global array is never global all at once:

In [31]:
explore.visibility_window_explorer()

Hawaii and Spain overlap for a couple of hours; Chile sees the source for most
of the night. Every baseline exists only during the intersection of two of these
strips, which is the real constraint on what gets measured.

Raising the elevation limit is roughly what bad weather does: low-elevation
observations look through more atmosphere, and dropping them costs baselines.

## The Earth's rotation does the rest of the work

Here's the trick that makes sparse arrays work. The baseline vector is fixed in
the Earth, but the Earth turns, so its *projection* onto the sky plane sweeps out
an arc over the night. Each pair of dishes measures not one Fourier
component but a whole track of them.

And because the sky is real, every measurement at $(u,v)$ comes with a free
conjugate at $(-u,-v)$, which is why these plots are always symmetric:

In [32]:
explore.coverage_explorer()

Turn ALMA off. Most of the plane empties, because ALMA anchors nearly every long
baseline that matters. Turn everything off except JCMT and SMA, which sit 164 m
apart on Mauna Kea, and you are left with one dot near the origin: a baseline
that measures the total flux and nothing else.

Even with all eight stations, look at how much of the plane is blank. The
unsampled regions correspond to real structures on the sky that the data simply
does not constrain. That's the inverse problem, visible.

## How fine a pixel is worth using

The coverage fixes the numbers you need to set up any imaging run. Both of them
come from the sampling theorem, the one that says how finely a signal has to be
sampled before it is pinned down, applied here to the $uv$ plane instead of to a
signal in time:

- **Resolution** $\approx 1/u_{\max}$, the finest fringe measured.
- **Field of view** $\approx 1/\Delta u$, where $\Delta u$ is the spacing between
  neighbouring samples. Structure wider than that does not go unmeasured, it gets
  *folded back in* at the wrong place, the same effect that makes a wheel in a
  film look like it is turning backwards.
- **Pixels** must be a few times finer than the resolution, or the model can't
  represent what the data constrains. This is a statement about the *array*, not
  the source.

In [33]:
coverage = arrays.uv_coverage(array, explore.MJD_2017_APR11, np.linspace(0, 24, 145),
                          *arrays.SOURCES["M87"])
u_max = np.hypot(coverage["u"], coverage["v"]).max()

print(f"samples collected      {len(coverage['u']):,}")
print(f"longest baseline       {u_max / 1e9:.2f} Gλ  "
      f"({u_max * 299792458 / 230e9 / 1e3:,.0f} km)")
print(f"nominal resolution     {itf.beam_size_uas(coverage['u'], coverage['v']):.1f} µas"
      "   <- vs a 42 µas ring: barely resolved")
print(f"Nyquist pixel (3×)     {itf.nyquist_pixel_uas(u_max):.2f} µas")

samples collected      840
longest baseline       8.37 Gλ  (10,907 km)
nominal resolution     24.7 µas   <- vs a 42 µas ring: barely resolved
Nyquist pixel (3×)     8.22 µas


"Barely resolved" is the honest description of the 2017 M87 result: the ring is
about two resolution elements across. Everything about how carefully those images
were validated follows from that number.

---
# Part C · What a station actually records

Up to here a "station" has been a point that produces a number. It isn't. It's a
dish with a receiver at the focus, and the receiver has **two feeds**. As
notebook 01 established, the electromagnetic field has two independent
components, and it takes two measurements to capture them.

## Two feeds, and therefore four numbers

Each feed projects the incoming field onto one direction and produces a voltage.
A circular-feed station projects onto $R$ and $L$; a linear-feed station projects
onto $X$ and $Y$. Nothing about the light differs, only the description.

In [34]:
sites = arrays.load_sites()
for code in ("ALMA", "SMA", "PDB", "JCMT"):
    site = sites[code]
    print(f"{code:5s} records {site.feeds[0].upper()} and {site.feeds[1].upper()}"
          f"   ({'linear' if site.is_linear else 'circular'} feeds)")

ALMA  records X and Y   (linear feeds)
SMA   records R and L   (circular feeds)
PDB   records X and Y   (linear feeds)
JCMT  records R and L   (circular feeds)


A **baseline** correlates every feed of one station against every feed of the
other, so each pair of dishes yields **four** numbers per moment, not one:

| both circular | both linear | one of each (*mixed*) |
|---|---|---|
| RR, LL, RL, LR | XX, YY, XY, YX | RX, RY, LX, LY |

The first two columns are what existing imaging software understands. The third
is what happens on every ALMA baseline, and it is the subject of notebook 03.

## The feeds turn against the sky

Here's the complication that makes polarimetry hard even in a single basis.

The feeds are bolted into a mount that sits on the ground. The source moves
across the sky. So over a night, the feed frame **rotates relative to the sky
frame**, by an angle the dish's own geometry decides.

Most of it is the **parallactic angle**: the angle between "up" as the mount sees
it and north as the sky sees it, which slides continuously as the source rises,
transits and sets. Some dishes add a second term. On one with **Nasmyth optics**
the receiver sits off to the side of the mount instead of at the focus, with a
mirror folding the beam into it, so the elevation turns the feeds as well, with a
sign that depends on which side the receiver is on.

$$\phi = f_{\rm par}\,\psi_{\rm par} + f_{\rm elev}\,\theta_{\rm elev} + \phi_0$$

Those coefficients are per-station properties that ship with the array table:

In [35]:
for code in ("JCMT", "SMA", "APEX", "PDB"):
    site = sites[code]
    mount = ("equatorial (no rotation)" if (site.fr_par, site.fr_elev) == (0, 0)
             else "alt-az" if site.fr_elev == 0 else "Nasmyth")
    print(f"{code:5s} φ = {site.fr_par:+.0f}·parallactic {site.fr_elev:+.0f}·elevation "
          f"{site.fr_offset_deg:+.0f}°   [{mount}]")

JCMT  φ = +1·parallactic +0·elevation +0°   [alt-az]
SMA   φ = +1·parallactic -1·elevation +45°   [Nasmyth]
APEX  φ = +1·parallactic +1·elevation +0°   [Nasmyth]
PDB   φ = +0·parallactic +0·elevation +0°   [equatorial (no rotation)]


In [36]:
explore.field_rotation_explorer()

Every station follows its own curve. This is the first point in the story where
polarization stops being a property of the source and becomes a property of the
*instrument*. Note how it lands differently on the two feed types:

- a **circular** feed picks up a *phase* $e^{\mp i\phi}$ from this rotation;
- a **linear** feed has its $Q$ and $U$ **mixed** by $2\phi$.

Same sky, same instant, two completely different corrections. Which is fine if
your array is all one type. Notebook 03 is about when it isn't.

## Noise belongs to a feed, not to a station

The numbers come with error bars, set by how noisy the two receivers are. That is
what their system equivalent flux densities (SEFD) measure:

$$\sigma \;\propto\; \sqrt{\mathrm{SEFD}_1 \cdot \mathrm{SEFD}_2}$$

The *geometric mean* of the two is the part worth remembering: it is why one very
sensitive station lifts every baseline it touches. (The full expression, in
`arrays.thermal_noise` and in eht-imaging, also divides by the bandwidth and
integration time being averaged over, and by 0.88 for two-bit recording.)

Now the subtlety. An SEFD is a property of a **feed**, not of a station. So what
happens to those four error bars when the two ends of a baseline have different
kinds of feed? Pick ALMA and a circular station and look at the labels:

In [37]:
explore.noise_explorer()

Pick ALMA and any other station and read the slot labels: XR, XL, YR, YL. Those
aren't correlations that any single-basis imaging pipeline knows how to
interpret, and getting their noise pairing right inside eht-imaging was part of
the work this report is about.

---
# Part D · From numbers to a picture

## The dirty image is the data, not the sky

The obvious thing to do with sampled Fourier components is to inverse-transform
them. Do that and you get the **dirty image**: the true sky convolved with the
transform of the sampling pattern, the **dirty beam**.

In [38]:
explore.imaging_explorer()

The middle panel is the point. Those rings and streaks aren't in the sky; they're
the array's fingerprint: the beam's **sidelobes**, meaning everything outside its
central peak. They are baked into every feature of the right-hand panel.
Drop stations and watch them grow into structure that looks entirely plausible
and is entirely fictional.

Since the unsampled Fourier components are unconstrained, **infinitely many
images fit the data exactly**: you can add any structure living only in the gaps
and change nothing measurable. Choosing among them requires an assumption
about what a sky is allowed to look like. That's not a flaw in the method; it's
the method.

## What survives bad calibration

There's a second problem, and at 230 GHz it's worse than the first. The
atmosphere above each station adds a phase that changes by radians in seconds.
The measured visibility isn't $V_{12}$, it's

$$V_{12}^{\rm obs} = g_1 g_2^{*}\, V_{12}$$

with unknown complex station gains. Individual visibility phases are garbage.

But add the phases around a *closed triangle* and every station appears once
with each sign:

$$\arg V^{\rm obs}_{12} + \arg V^{\rm obs}_{23} + \arg V^{\rm obs}_{31}
 = \arg V_{12} + \arg V_{23} + \arg V_{31}$$

The corruption cancels **exactly**. Drag the noise up to 180° and watch:

In [39]:
explore.closure_explorer()

The top panel becomes noise; the bottom panel doesn't move. The EHT's first
images were made almost entirely from quantities with this property.

Note the shape of the trick, because it's about to fail us: closure phases
survive **station-based** corruptions. The corruptions in Part C, feed rotation
and the leakage terms coming in notebook 03, are also station based, but they act
on *polarization*, mixing the four correlation products into each other. A
closure phase built from a single product doesn't protect you there. Polarimetry
has to model the corruption, not dodge it.

## Choosing among the images that fit

Modern EHT imaging (and eht-imaging specifically) picks among the infinitely many
fitting images by minimizing

$$\underbrace{\chi^2(\text{data}, I)}_{\text{fit the measurements}} \;+\;
  \sum_i \alpha_i\, R_i(I)$$

where the $R_i$ are **regularizers**: preferences about what a sky is allowed to
look like: smooth patches with sharp edges between them, closeness to a prior
image, a known total flux, brightness that is never negative. The $\alpha_i$ set
how much you're willing to let the prior speak where the data is silent. Too
little and you reconstruct the beam's sidelobes; too much and you erase real
structure. The EHT collaboration's answer to "how do we know we chose right" was
to survey the whole parameter space and require the result to be stable across
it.

The $\chi^2$ can be measured against whichever data products survived
calibration (complex visibilities if any did, otherwise amplitudes and closure
quantities), and eht-imaging implements every one of them for each of three ways
of turning an image into visibilities (`direct`, `fast`, `nfft`). That product is
a large part of why the module is 4,000 lines long, and why it is being
restructured.

## Why all of it has to be differentiable

Minimizing that objective means differentiating the whole chain (image → Fourier
transform → sampled visibilities → $\chi^2$) with respect to every pixel. That
constrains how the forward model may be written: every step has to be
differentiable.

It pays off if you push it further. Make the calibration terms (station gains,
leakage, feed rotation) parameters with gradients too, and the image and the
calibration can be solved **jointly**. The usual alternative is to alternate:
image, then **self-calibrate** (fit the station gains against the image you just
made), then re-image with those gains, and hope the loop settles somewhere
sensible. Solving both at once instead needs one differentiable chain from every
parameter down to the loss, which is what JAX is for, and it is why eht-imaging
is being restructured into pure functional backends.

The innermost piece of that chain is the forward model itself: the step that
turns a Stokes image into the four correlation products a baseline actually
records.
Notebook 03 builds that step, for the awkward case where the two ends of the
baseline disagree about what they are recording.

## Recap

- **$\theta \sim \lambda/D$** forces an Earth-sized aperture, so the EHT builds
  one out of pairs of dishes.
- Each pair measures **one Fourier component** of the sky (van Cittert–Zernike),
  at a $(u,v)$ set by the baseline projected onto the sky plane.
- **Earth rotation** sweeps each baseline into an arc, and conjugate symmetry
  doubles the samples, but the $uv$ plane stays mostly empty, so the inverse
  problem is underdetermined and needs priors.
- Nyquist ties the array to the image: resolution $=1/u_{\max}$,
  field of view $=1/\Delta u$.
- A station records **two feed voltages**, in a circular or a linear basis, on a
  mount whose feeds **rotate against the sky**, differently at each station.
  Each baseline therefore produces **four** correlation products, with noise set
  per feed pair.
- **Closure quantities** are immune to station-based *gain* errors, which is what
  made VLBI imaging possible, but they don't rescue polarimetry, which has to
  model its corruptions instead.

Everything above quietly assumed both ends of every baseline speak the same
polarization language. On any ALMA baseline, they don't.

**Next:** [03 · Jones matrices and mixed
polarization](03_jones_and_mixed_polarization.ipynb). One 2×2 matrix describes
everything between the sky and the recorded number, and that notebook is about
what happens when the two ends of a baseline disagree about what they are
recording.